# 02 — Baseline and group-safe validation

**Purpose:** Establish a reproducible no-image benchmark before vision training.

The starter baseline predicts the **mean training distribution**. Its leave-one-out evaluation always holds out a full soil sample, never an individual photograph. The baseline is not expected to win the competition.

This notebook assumes the raw labels have already been checked. If their schema differs, adapt the loader in `src/baseline.py` first.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.baseline import read_labels, loocv_mean_baseline, write_submission
from src.metric import TARGET_COLUMNS, DIAMETERS, LOG_WEIGHTS, validate_curves, mean_emd
LABELS_PATH = ROOT / "data" / "Training_labels_updated.csv"
SUBMISSION_TEMPLATE = ROOT / "data" / "sample_submission.csv"
print("Labels exist:", LABELS_PATH.exists(), "| template exists:", SUBMISSION_TEMPLATE.exists())

## 1. Verify logarithmic EMD behavior

The exact competition scoring convention should be cross-checked against organizer code if available.

In [ ]:
toy = np.array([0, 0, 0, 5, 12, 40, 70, 90, 99, 100, 100], dtype=float)
assert mean_emd(toy, toy) == 0
assert np.all(LOG_WEIGHTS > 0)
print("Perfect-score sanity check passed")
print("Sum of log interval weights:", LOG_WEIGHTS.sum())

## 2. Run leave-one-soil-sample-out validation

This notebook intentionally stops before generating scores if the official labels are not present.

In [ ]:
if LABELS_PATH.exists():
    labels = read_labels(LABELS_PATH)
    score = loocv_mean_baseline(labels)
    print(f"Labeled soil samples: {len(labels)}")
    print(f"LOO EMD: {score:.6f} (lower is better)")
else:
    labels = None
    print("Download and inspect the official dataset first; no result computed.")

## 3. Plot the mean training curve

In [ ]:
if labels is not None:
    y = labels[TARGET_COLUMNS].to_numpy(dtype=float)
    validate_curves(y)
    fig, ax = plt.subplots(figsize=(8, 5))
    for row in y:
        ax.plot(DIAMETERS, row, alpha=.2, color="gray")
    ax.plot(DIAMETERS, y.mean(axis=0), linewidth=3, label="Mean baseline")
    ax.set_xscale("log")
    ax.set(xlabel="Diameter (mm)", ylabel="Cumulative mass (%)", ylim=(0,100))
    ax.grid(alpha=.2)
    ax.legend()
    plt.show()

## 4. Write a Kaggle submission (only after schema checks)

The writer follows the supplied template's column order. This notebook does not submit anything to Kaggle.

In [ ]:
if labels is not None and SUBMISSION_TEMPLATE.exists():
    output = ROOT / "submissions" / "mean_baseline.csv"
    write_submission(labels, SUBMISSION_TEMPLATE, output)
    print("Review this CSV locally before uploading:", output)
else:
    print("Submission not created: labels or official template missing.")

## Next experiments

1. Audit photo-to-sample and camera metadata joins.
2. Extract frozen pretrained visual embeddings per photograph.
3. Aggregate photographs within soil samples.
4. Evaluate regularized models with the same sample-level folds.
5. Compare with the mean baseline before submitting.